In [41]:
import pandas as pd
import os
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [42]:
base_dir = os.path.join(os.getcwd(),'src')
train_dir = os.path.join(base_dir,'train.csv')
test_dir = os.path.join(base_dir,'test.csv')

le=LabelEncoder()

In [43]:
df_train = pd.read_csv(train_dir)
df_test = pd.read_csv(test_dir)

In [44]:
df_train.head

<bound method NDFrame.head of             id health_condition  sleep_duration  heart_rate    bmi  \
0            0        unhealthy            5.22        70.6  25.66   
1            1          at-risk            5.53        71.3  25.84   
2            2        unhealthy            5.29        75.4  24.54   
3            3        unhealthy            4.70        77.2  23.13   
4            4          at-risk            7.23        73.4  28.44   
...        ...              ...             ...         ...    ...   
690083  690083          at-risk            6.31        69.7  24.11   
690084  690084          at-risk            5.78        54.0  26.36   
690085  690085              fit            7.64        85.7  21.91   
690086  690086          at-risk            6.74        73.0  18.73   
690087  690087          at-risk            5.55        69.3  24.38   

        calorie_expenditure  step_count  exercise_duration  water_intake  \
0                    2174.0      1326.0              

In [45]:
df_train.columns

Index(['id', 'health_condition', 'sleep_duration', 'heart_rate', 'bmi',
       'calorie_expenditure', 'step_count', 'exercise_duration',
       'water_intake', 'diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender'],
      dtype='str')

In [46]:
df_train.dtypes

id                           int64
health_condition               str
sleep_duration             float64
heart_rate                 float64
bmi                        float64
calorie_expenditure        float64
step_count                 float64
exercise_duration          float64
water_intake               float64
diet_type                      str
stress_level                   str
sleep_quality                  str
physical_activity_level        str
smoking_alcohol                str
gender                         str
dtype: object

In [47]:
str_col = df_train.select_dtypes(include=['str']).columns

for col in str_col:
    print(f"----カラム名：{col}---")
    print(df_train[col].unique())
    print()

----カラム名：health_condition---
<StringArray>
['unhealthy', 'at-risk', 'fit']
Length: 3, dtype: str

----カラム名：diet_type---
<StringArray>
['veg', 'non-veg', 'balanced', nan]
Length: 4, dtype: str

----カラム名：stress_level---
<StringArray>
['high', 'low', nan, 'medium']
Length: 4, dtype: str

----カラム名：sleep_quality---
<StringArray>
['average', 'poor', nan, 'good']
Length: 4, dtype: str

----カラム名：physical_activity_level---
<StringArray>
['sedentary', 'moderate', 'active', nan]
Length: 4, dtype: str

----カラム名：smoking_alcohol---
<StringArray>
['yes', 'occasional', nan, 'no']
Length: 4, dtype: str

----カラム名：gender---
<StringArray>
['female', 'other', 'male', nan]
Length: 4, dtype: str



In [48]:
tr_col = df_train.select_dtypes(include=['str']).columns

for col in str_col:
    print(f"----カラム名：{col}---")
    print(df_train[col].value_counts(dropna=False))
    print()

----カラム名：health_condition---
health_condition
at-risk      592561
unhealthy     57724
fit           39803
Name: count, dtype: int64

----カラム名：diet_type---
diet_type
veg         231432
balanced    226888
non-veg     224867
NaN           6901
Name: count, dtype: int64

----カラム名：stress_level---
stress_level
medium    261819
high      177750
low       167708
NaN        82811
Name: count, dtype: int64

----カラム名：sleep_quality---
sleep_quality
average    213948
poor       212166
good       205643
NaN         58331
Name: count, dtype: int64

----カラム名：physical_activity_level---
physical_activity_level
moderate     221041
sedentary    219784
active       212642
NaN           36621
Name: count, dtype: int64

----カラム名：smoking_alcohol---
smoking_alcohol
yes           223730
no            219791
occasional    217985
NaN            28582
Name: count, dtype: int64

----カラム名：gender---
gender
male      237756
female    224016
other     206943
NaN        21373
Name: count, dtype: int64



In [ ]:
str_col = df_test.select_dtypes(include=['str']).columns

for col in str_col:
    print(f"----testカラム名：{col}---")
    print(df_test[col].unique())
    print()
    print(f"----trainカラム名：{col}---")
    print(df_train[col].unique())
    print()

----testカラム名：diet_type---
<StringArray>
['veg', 'balanced', 'non-veg', nan]
Length: 4, dtype: str

----trainカラム名：diet_type---
<StringArray>
['veg', 'non-veg', 'balanced', nan]
Length: 4, dtype: str

----testカラム名：stress_level---
<StringArray>
['high', 'medium', 'low', nan]
Length: 4, dtype: str

----trainカラム名：stress_level---
<StringArray>
['high', 'low', nan, 'medium']
Length: 4, dtype: str

----testカラム名：sleep_quality---
<StringArray>
['poor', 'good', 'average', nan]
Length: 4, dtype: str

----trainカラム名：sleep_quality---
<StringArray>
['average', 'poor', nan, 'good']
Length: 4, dtype: str

----testカラム名：physical_activity_level---
<StringArray>
['active', 'sedentary', 'moderate', nan]
Length: 4, dtype: str

----trainカラム名：physical_activity_level---
<StringArray>
['sedentary', 'moderate', 'active', nan]
Length: 4, dtype: str

----testカラム名：smoking_alcohol---
<StringArray>
['occasional', 'yes', 'no', nan]
Length: 4, dtype: str

----trainカラム名：smoking_alcohol---
<StringArray>
['yes', 'occasional

In [53]:
cat_cols = [
    'health_condition', 'diet_type', 'stress_level', 
    'sleep_quality', 'physical_activity_level', 
    'smoking_alcohol', 'gender'
]

In [ ]:
X=df_train.drop(columns=['health_condition'])
y=df_train['health_condition']
X_test=df_test.copy()
y=le.fit_transform(df_train['health_condition'])

for i,class_name in enumerate(le.classes_):
    print(f"{class_name} -> {i}")

In [52]:
for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')

KeyError: 'health_condition'

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
model = lgb.LGBMClassifier(random_state=42)
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)]
)

In [ ]:
preds = model.predict(X_test)